In [1]:
!pip install -q transformers torch deep-translator spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 31.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
import requests
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from deep_translator import GoogleTranslator

# 1. Load Deep Learning Grammar Model from Hugging Face
print("🔄 Loading Grammar Correction Model (T5)...")
grammar_tokenizer = AutoTokenizer.from_pretrained("vennify/t5-base-grammar-correction")
grammar_model = AutoModelForSeq2SeqLM.from_pretrained("vennify/t5-base-grammar-correction")

# 2. Load spaCy for Study Mode (Vocabulary extraction)
print("🔄 Loading spaCy NLP Engine...")
nlp = spacy.load("en_core_web_sm")
print("✅ All Models Loaded Successfully!\n")

def correct_grammar(text):
    """Corrects English grammar using a fine-tuned T5 Transformer model."""
    input_text = "gec: " + text
    inputs = grammar_tokenizer(input_text, return_tensors="pt", padding=True)
    outputs = grammar_model.generate(**inputs, max_length=128)
    corrected_text = grammar_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_text

def translate_text(text, target_lang_code):
    """Translates text using deep-translator (e.g., 'ta' for Tamil, 'fr' for French)."""
    translated = GoogleTranslator(source='auto', target=target_lang_code).translate(text)
    return translated

def study_mode_features(text):
    """Extracts difficult vocabulary words and fetches definitions using a dictionary API."""
    doc = nlp(text)
    # Extract nouns, verbs, and adjectives that aren't common stopwords
    important_words = list(set([token.lemma_.lower() for token in doc if not token.is_stop and token.is_alpha and len(token.text) > 4]))

    study_notes = []
    for word in important_words[:3]:  # Limit to top 3 words
        try:
            response = requests.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{word}")
            if response.status_code == 200:
                data = response.json()
                definition = data[0]['meanings'][0]['definitions'][0]['definition']
                study_notes.append(f"  • {word.capitalize()}: {definition}")
        except:
            continue

    return study_notes

def run_nlp_pipeline(input_text, target_language="ta"):
    """Orchestrates the entire NLP pipeline and prints structured results."""
    print("=" * 60)
    print(f"📥 ORIGINAL INPUT: '{input_text}'")
    print("=" * 60)

    # Step 1: Grammar Correction
    corrected = correct_grammar(input_text)
    print(f"✨ AI CORRECTED:   {corrected}")

    # Step 2: Translation (using the corrected text)
    translated = translate_text(corrected, target_language)
    lang_name = "Tamil" if target_language == "ta" else target_language.upper()
    print(f"🌐 {lang_name} TRANSLATION: {translated}")

    # Step 3: Study Mode (Vocabulary Insights)
    print("\n📚 STUDY MODE INSIGHTS:")
    definitions = study_mode_features(corrected)
    if definitions:
        for item in definitions:
            print(item)
    else:
        print("  • No complex vocabulary detected.")
    print("=" * 60 + "\n")

🔄 Loading Grammar Correction Model (T5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🔄 Loading spaCy NLP Engine...
✅ All Models Loaded Successfully!



In [3]:
# Example 1: Testing with broken grammar and translating to Tamil ('ta')
run_nlp_pipeline(
    input_text="She dont likes the complicated assignments",
    target_language="ta"
)

# Example 2: Testing another sentence and translating to French ('fr')
run_nlp_pipeline(
    input_text="He go to school yesterday and see a enormous bird",
    target_language="fr"
)

📥 ORIGINAL INPUT: 'She dont likes the complicated assignments'
✨ AI CORRECTED:   She doesn't like the complicated assignments.
🌐 Tamil TRANSLATION: சிக்கலான பணிகளை அவள் விரும்புவதில்லை.

📚 STUDY MODE INSIGHTS:
  • Assignment: The act of assigning; the allocation of a job or a set of tasks.
  • Complicated: To make complex; to modify so as to make something intricate or difficult.

📥 ORIGINAL INPUT: 'He go to school yesterday and see a enormous bird'
✨ AI CORRECTED:   He went to school yesterday and saw an enormous bird.
🌐 FR TRANSLATION: Il est allé à l'école hier et a vu un énorme oiseau.

📚 STUDY MODE INSIGHTS:
  • School: (collective) A group of fish or a group of marine mammals such as porpoises, dolphins, or whales.
  • Yesterday: The day immediately before today; one day ago.
  • Enormous: Deviating from the norm; unusual, extraordinary.

